# Prepare a PubChem assay dataset

Download one BioAssay as active and inactive SDF files in the folder expected by `PubChemAssayLoader`. This keeps raw assay data out of Git while making the workflow repeatable.

Set the assay ID and optional per-class limit below. The default prepares up to 1,000 compounds per class for a quick workflow; set the limit to `None` to download every compound. Large assays can take several minutes.

Output: `data-local/PUBCHEM/AID<id>_active.sdf`, `AID<id>_inactive.sdf`, and a small JSON manifest. The loader caches parsed graphs under `data-local/graph_corpus_cache/`.

PubChem endpoints used: [PUG-REST assay summary](https://pubchem.ncbi.nlm.nih.gov/docs/pug-rest) and the [PubChem download guide](https://pubchem.ncbi.nlm.nih.gov/docs/downloads).


In [ ]:
from pathlib import Path
import runpy

BOOTSTRAP_CANDIDATES = (
    "notebooks/_bootstrap.py",
    "abstractgraph-graphicalizer/notebooks/_bootstrap.py",
)
for candidate in BOOTSTRAP_CANDIDATES:
    if Path(candidate).exists():
        _bootstrap = runpy.run_path(candidate)
        break
else:
    raise FileNotFoundError("Could not find notebooks/_bootstrap.py")

repo_root = _bootstrap["repo_root"]
workspace_root = _bootstrap["workspace_root"]


## Choose the assay

Use an AID from [PubChem BioAssay](https://pubchem.ncbi.nlm.nih.gov/bioassay/624249) or another assay page. `MAX_CIDS_PER_ACTIVITY` limits active and inactive sets equally; use `None` for the complete assay.


In [ ]:
ASSAY_ID = "624249"
MAX_CIDS_PER_ACTIVITY = 1_000  # use None to download the full assay
BATCH_SIZE = 100
OVERWRITE = False

output_root = repo_root / "data-local" / "PUBCHEM"
output_root.mkdir(parents=True, exist_ok=True)
print("Assay:", ASSAY_ID)
print("Output:", output_root)


## Fetch active and inactive structures

The notebook requests the assay’s active and inactive compound IDs, retrieves structures in small batches, and writes each class in the SDF layout consumed by `PubChemAssayLoader`. Existing files are reused unless `OVERWRITE=True`. Set `OVERWRITE=True` if you change the per-class limit and want to replace existing SDF files.


In [ ]:
from datetime import datetime, timezone
import json
import os
import time

import requests

BASE_URL = "https://pubchem.ncbi.nlm.nih.gov/rest/pug"
session = requests.Session()
session.headers.update({"User-Agent": "AbstractGraph-data-preparation/1.0"})

def get_json(path, params=None):
    url = f"{BASE_URL}/{path}"
    for attempt in range(6):
        response = session.get(url, params=params, timeout=90)
        if response.status_code in {429, 500, 502, 503, 504} and attempt < 5:
            time.sleep(min(2 ** attempt, 30))
            continue
        response.raise_for_status()
        return response.json()
    raise RuntimeError(f"PubChem request failed: {url}")

def get_bytes(path):
    url = f"{BASE_URL}/{path}"
    for attempt in range(6):
        response = session.get(url, timeout=120)
        if response.status_code in {429, 500, 502, 503, 504} and attempt < 5:
            time.sleep(min(2 ** attempt, 30))
            continue
        response.raise_for_status()
        return response.content
    raise RuntimeError(f"PubChem request failed: {url}")

def assay_cids(assay_id, activity):
    payload = get_json(f"assay/aid/{assay_id}/cids/JSON", {"cids_type": activity})
    records = payload.get("InformationList", {}).get("Information", [])
    cids = records[0].get("CID", []) if records else []
    cids = list(dict.fromkeys(int(cid) for cid in cids))
    if MAX_CIDS_PER_ACTIVITY is not None:
        cids = cids[: int(MAX_CIDS_PER_ACTIVITY)]
    return cids

def download_sdf(assay_id, activity, cids, output_path):
    if not cids:
        raise ValueError(f"PubChem returned no {activity} CIDs for AID {assay_id}")
    temporary_path = output_path.with_suffix(output_path.suffix + ".part")
    downloaded_records = 0
    try:
        with temporary_path.open("wb") as output:
            for start in range(0, len(cids), BATCH_SIZE):
                batch = cids[start : start + BATCH_SIZE]
                cid_text = ",".join(str(cid) for cid in batch)
                content = get_bytes(f"compound/cid/{cid_text}/SDF")
                record_count = content.count(b"$$$$")
                if record_count == 0:
                    raise RuntimeError(f"No SDF records returned for {activity} batch starting at {start}")
                output.write(content)
                downloaded_records += record_count
                time.sleep(0.25)
        os.replace(temporary_path, output_path)
    except Exception:
        temporary_path.unlink(missing_ok=True)
        raise
    return downloaded_records

assay_id = str(ASSAY_ID).removeprefix("AID")
summary = get_json(f"assay/aid/{assay_id}/summary/JSON")
summary_rows = summary.get("AssaySummaries", {}).get("AssaySummary", [])
assay_title = summary_rows[0].get("Name", "") if summary_rows else ""
manifest = {
    "assay_id": assay_id,
    "assay_title": assay_title,
    "source": f"PubChem BioAssay AID {assay_id}",
    "source_url": f"https://pubchem.ncbi.nlm.nih.gov/bioassay/{assay_id}",
    "retrieved_utc": datetime.now(timezone.utc).isoformat(),
    "max_cids_per_activity": MAX_CIDS_PER_ACTIVITY,
    "batch_size": BATCH_SIZE,
    "classes": {},
}

for activity in ("active", "inactive"):
    output_path = output_root / f"AID{assay_id}_{activity}.sdf"
    if output_path.exists() and not OVERWRITE:
        print(f"Using existing {output_path}")
        manifest["classes"][activity] = {"path": output_path.name, "reused": True}
        continue
    cids = assay_cids(assay_id, activity)
    record_count = download_sdf(assay_id, activity, cids, output_path)
    manifest["classes"][activity] = {
        "path": output_path.name,
        "requested_cids": len(cids),
        "downloaded_sdf_records": record_count,
        "reused": False,
    }
    print(f"Saved {record_count} {activity} structures to {output_path}")

if assay_title:
    (output_root / f"AID{assay_id}_info.txt").write_text(assay_title + "\n")
(output_root / f"AID{assay_id}_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")
manifest


## Continue with analysis

Once the two SDF files exist, run [`example_pubchem_loader_statistics.ipynb`](../examples/example_pubchem_loader_statistics.ipynb) to inspect the prepared assay. Downstream ML notebooks can then use `PubChemAssayLoader` with the same AID.
